In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Create SparkSession
spark = SparkSession.builder \
    .appName("Advanced Transformations") \
    .master("local[*]") \
    .getOrCreate()

# Create sample DataFrame
data = [
    ("Alice", "Sales", 50000, "2024-01"),
    ("Bob", "IT", 60000, "2024-01"),
    ("Charlie", "Sales", 70000, "2024-01"),
    ("Diana", "IT", 55000, "2024-01"),
    ("Alice", "Sales", 52000, "2024-02"),
    ("Bob", "IT", 61000, "2024-02"),
    ("Charlie", "Sales", 72000, "2024-02"),
    ("Diana", "IT", 56000, "2024-02")
]

schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Month", StringType(), True)
])

df = spark.createDataFrame(data, schema)
df.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/05 14:35:33 WARN Utils: Your hostname, Ahyaans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.17 instead (on interface en0)
26/01/05 14:35:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/05 14:35:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+----------+------+-------+
|   Name|Department|Salary|  Month|
+-------+----------+------+-------+
|  Alice|     Sales| 50000|2024-01|
|    Bob|        IT| 60000|2024-01|
|Charlie|     Sales| 70000|2024-01|
|  Diana|        IT| 55000|2024-01|
|  Alice|     Sales| 52000|2024-02|
|    Bob|        IT| 61000|2024-02|
|Charlie|     Sales| 72000|2024-02|
|  Diana|        IT| 56000|2024-02|
+-------+----------+------+-------+



In [3]:
# Define a window partitioned by Department, ordered by Salary
window_spec = Window.partitionBy("Department").orderBy(col("Salary").desc())

# Rank employees within each department
df_with_rank = df.withColumn("Rank", rank().over(window_spec))
df_with_rank.show()

+-------+----------+------+-------+----+
|   Name|Department|Salary|  Month|Rank|
+-------+----------+------+-------+----+
|    Bob|        IT| 61000|2024-02|   1|
|    Bob|        IT| 60000|2024-01|   2|
|  Diana|        IT| 56000|2024-02|   3|
|  Diana|        IT| 55000|2024-01|   4|
|Charlie|     Sales| 72000|2024-02|   1|
|Charlie|     Sales| 70000|2024-01|   2|
|  Alice|     Sales| 52000|2024-02|   3|
|  Alice|     Sales| 50000|2024-01|   4|
+-------+----------+------+-------+----+



In [ ]:
window_sum = Window.partitionBy("Department").orderBy("Month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_with_running_total = df.withColumn("RunningTotal", sum("Salary").over(window_sum))
df_with_running_total.show()

# .rowBetween(window.unboundedPreceding, window.currentRow)


+-------+----------+------+-------+------------+
|   Name|Department|Salary|  Month|RunningTotal|
+-------+----------+------+-------+------------+
|    Bob|        IT| 60000|2024-01|       60000|
|  Diana|        IT| 55000|2024-01|      115000|
|    Bob|        IT| 61000|2024-02|      176000|
|  Diana|        IT| 56000|2024-02|      232000|
|  Alice|     Sales| 50000|2024-01|       50000|
|Charlie|     Sales| 70000|2024-01|      120000|
|  Alice|     Sales| 52000|2024-02|      172000|
|Charlie|     Sales| 72000|2024-02|      244000|
+-------+----------+------+-------+------------+



In [10]:
# Calculate average salary per department (using window)
window_avg = Window.partitionBy("Department").orderBy("Salary").rowsBetween(Window.unboundedPreceding,Window.currentRow)

df_with_avg = df.withColumn("AvgSalary", avg("Salary").over(window_avg))
df_with_avg.show()

+-------+----------+------+-------+------------------+
|   Name|Department|Salary|  Month|         AvgSalary|
+-------+----------+------+-------+------------------+
|  Diana|        IT| 55000|2024-01|           55000.0|
|  Diana|        IT| 56000|2024-02|           55500.0|
|    Bob|        IT| 60000|2024-01|           57000.0|
|    Bob|        IT| 61000|2024-02|           58000.0|
|  Alice|     Sales| 50000|2024-01|           50000.0|
|  Alice|     Sales| 52000|2024-02|           51000.0|
|Charlie|     Sales| 70000|2024-01|57333.333333333336|
|Charlie|     Sales| 72000|2024-02|           61000.0|
+-------+----------+------+-------+------------------+



In [ ]:
window_spec_lag_lead = Window.partitionBy("Department").orderBy("Month")